# Session 3 — Solutions

Worked solutions for signal region, cutflow, yields, and control regions (Z CR, Top CR). Uses one file per dataset from `config/datasets_2017_short.yaml`; SR plots are MC-only; data can be plotted in CRs.

## Setup and cutflow (Ex 3.1, 3.3)


In [ ]:
# Ensure project root is on sys.path (for SWAN / any kernel)
import sys, os
sys.path.append("..")


import numpy as np
import awkward as ak
import matplotlib.pyplot as plt
import pandas as pd

from config.datasets_2017 import get_short_fileset_and_labels
from processor.bbdm_processor import get_nanoevents, select_good_jets, count_bjets, n_tight_leptons, MET_SR_MIN

# Load one file per dataset from config/datasets_2017_short.yaml (limit to 10k events per file)
MAX_EVENTS = 10_000
fileset, labels = get_short_fileset_and_labels()
events_by_dataset = {}
for name, path_list in fileset.items():
    if path_list:
        events_by_dataset[name] = get_nanoevents(path_list[0])[:MAX_EVENTS]
bkg_names = [k for k in fileset if "Run2017" not in k and "bbDM" not in k]

# Matplotlib legend labels (LaTeX)
latex_labels = {
    "DYJets": r"$Z(\ell\ell)+$jets",
    "ZJets": r"$Z(\nu\bar{\nu})+$jets",
    "WJets": r"$W(\ell\nu)+$jets",
    "DIBOSON": r"WW",
    "STop": r"Single $t$",
    "Top": r"$t\bar{t}$",
    "QCD": r"QCD",
    "SMH": r"SMH",
    "bbDM_2HDMa_LO_5f": r"bbDM (2HDM+a)",
    "MET_Run2017B": r"Data",
}

# Cutflow per dataset (Ex 3.1)
def run_cutflow(events):
    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)
    nbjets = count_bjets(good_jets)
    nlep = n_tight_leptons(events)
    met = events.MET.pt
    n1 = ak.sum(njets >= 1)
    n2 = ak.sum((njets >= 1) & (nbjets >= 2))
    n3 = ak.sum((njets >= 1) & (nbjets >= 2) & (nlep == 0))
    n4 = ak.sum((njets >= 1) & (nbjets >= 2) & (nlep == 0) & (met > MET_SR_MIN))
    return {"presel": int(n1), "≥2b": int(n2), "0lep": int(n3), "MET>200": int(n4)}

cutflow_by_dataset = {name: run_cutflow(ev) for name, ev in events_by_dataset.items()}
for name in cutflow_by_dataset:
    print(labels.get(name, name), cutflow_by_dataset[name])

# Yield table (Ex 3.3): one row per cut, one column per process
cuts = ["≥1 jet", "≥2 b-jets", "0 leptons", "MET>200 GeV"]
keys = ["presel", "≥2b", "0lep", "MET>200"]
data = {"Cut": cuts}
for name in list(events_by_dataset.keys())[:6]:
    data[labels.get(name, name)] = [cutflow_by_dataset[name][k] for k in keys]
df = pd.DataFrame(data)
print(df)

## MET in signal region — MC only (Ex 3.2)


In [ ]:
# MET in SR: MC-only stacked (no data in signal region)
bins_met = np.linspace(200, 600, 50)
met_arrays, leg_labels = [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    ev = events_by_dataset[name]
    good_jets = select_good_jets(ev)
    njets = ak.num(good_jets)
    nbjets = count_bjets(good_jets)
    nlep = n_tight_leptons(ev)
    met = ev.MET.pt
    sr_mask = (njets >= 1) & (nbjets >= 2) & (nlep == 0) & (met > MET_SR_MIN)
    met_sr = met[sr_mask]
    flat = ak.to_numpy(ak.ravel(met_sr))
    if len(flat) > 0:
        met_arrays.append(flat)
        leg_labels.append(latex_labels.get(name, labels.get(name, name)))
if met_arrays:
    plt.hist(met_arrays, bins=bins_met, label=leg_labels, stacked=True, histtype="stepfilled", alpha=0.7)
plt.xlabel("MET [GeV]")
plt.ylabel("Events")
plt.title("MET in signal region (MC only)")
plt.legend()
plt.show()

## Z control region (dilepton) — data and MC

Preselection **recoil** > 250 GeV (U = −(pTmiss + Σ pT^lep)), ≥1 jet, leading jet pT > 100 GeV; then exactly 2 OSSF leptons (ee or μμ), leading lepton pT > 30 GeV, 70 < m_ℓℓ < 110 GeV.


In [ ]:
from processor.bbdm_processor import select_tight_electrons, select_tight_muons, recoil_pt

PRESEL_RECOIL_MIN = 250.0
LEAD_JET_PT_MIN = 100.0
Z_CR_MLL_LO, Z_CR_MLL_HI = 70.0, 110.0
LEAD_LEP_PT_CR = 30.0

def z_cr_mask(events):
    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)
    lead_jet_pt = ak.fill_none(ak.firsts(good_jets.pt), 0.0)
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele = select_tight_electrons(events)
    tight_mu = select_tight_muons(events)
    nele, nmu = ak.count(tight_ele.pt, axis=1), ak.count(tight_mu.pt, axis=1)
    two_ee = (nele == 2) & (nmu == 0) & (ak.sum(tight_ele.charge, axis=1) == 0)
    two_mumu = (nele == 0) & (nmu == 2) & (ak.sum(tight_mu.charge, axis=1) == 0)
    tight_ele_pad = ak.pad_none(tight_ele, 2)
    tight_mu_pad = ak.pad_none(tight_mu, 2)
    pair_ee = tight_ele_pad[:, 0] + tight_ele_pad[:, 1]
    pair_mumu = tight_mu_pad[:, 0] + tight_mu_pad[:, 1]
    sum_lep_px = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.cos(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.cos(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    sum_lep_py = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.sin(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.sin(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    recoil = recoil_pt(met_pt, met_phi, sum_lep_px, sum_lep_py)
    presel = (recoil > PRESEL_RECOIL_MIN) & (njets >= 1) & (lead_jet_pt > LEAD_JET_PT_MIN)
    mll_ee = ak.where(two_ee, ak.fill_none(pair_ee.mass, -1.0), ak.full_like(met_pt, -1.0))
    mll_mumu = ak.where(two_mumu, ak.fill_none(pair_mumu.mass, -1.0), ak.full_like(met_pt, -1.0))
    mll = ak.where(two_ee, mll_ee, ak.where(two_mumu, mll_mumu, ak.full_like(met_pt, -1.0)))
    lead_lep_pt = ak.where(two_ee, ak.max(tight_ele.pt, axis=1), ak.where(two_mumu, ak.max(tight_mu.pt, axis=1), ak.full_like(met_pt, 0.0)))
    return presel & (two_ee | two_mumu) & (lead_lep_pt > LEAD_LEP_PT_CR) & (mll > Z_CR_MLL_LO) & (mll < Z_CR_MLL_HI)

def z_cr_recoil(events):
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele, tight_mu = select_tight_electrons(events), select_tight_muons(events)
    nele, nmu = ak.count(tight_ele.pt, axis=1), ak.count(tight_mu.pt, axis=1)
    two_ee = (nele == 2) & (nmu == 0) & (ak.sum(tight_ele.charge, axis=1) == 0)
    two_mumu = (nele == 0) & (nmu == 2) & (ak.sum(tight_mu.charge, axis=1) == 0)
    tight_ele_pad = ak.pad_none(tight_ele, 2)
    tight_mu_pad = ak.pad_none(tight_mu, 2)
    pair_ee = tight_ele_pad[:, 0] + tight_ele_pad[:, 1]
    pair_mumu = tight_mu_pad[:, 0] + tight_mu_pad[:, 1]
    sum_lep_px = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.cos(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.cos(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    sum_lep_py = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.sin(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.sin(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    return recoil_pt(met_pt, met_phi, sum_lep_px, sum_lep_py)

fig, ax = plt.subplots()
for name in list(events_by_dataset.keys())[:5]:
    ev = events_by_dataset[name]
    mask = z_cr_mask(ev)
    recoil = z_cr_recoil(ev)[mask]
    if len(ak.ravel(recoil)) == 0:
        continue
    ax.hist(ak.to_numpy(ak.ravel(recoil)), bins=30, range=(250, 600), label=latex_labels.get(name, labels.get(name, name)), alpha=0.6, histtype="step")
ax.set_xlabel("Recoil [GeV]")
ax.set_ylabel("Events")
ax.set_title("Z CR (dilepton): Recoil (data and MC)")
ax.legend()
plt.show()

## Top control region (single-lepton) — data and MC

Exactly one lepton pT > 30 GeV; m_T < 160 GeV; ≥2 b-jets; ≥2 non-b jets.

In [ ]:
TOP_CR_MT_MAX = 160.0

def top_cr_mask(events):
    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)
    nbjets = count_bjets(good_jets)
    lead_jet_pt = ak.fill_none(ak.firsts(good_jets.pt), 0.0)
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele = select_tight_electrons(events)
    tight_mu = select_tight_muons(events)
    one_ele = (ak.count(tight_ele.pt, axis=1) == 1) & (ak.count(tight_mu.pt, axis=1) == 0)
    one_mu = (ak.count(tight_ele.pt, axis=1) == 0) & (ak.count(tight_mu.pt, axis=1) == 1)
    lep_pt = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.pt), ak.where(one_mu, ak.firsts(tight_mu.pt), ak.full_like(met_pt, 0.0))), 0.0)
    lep_phi = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.phi), ak.where(one_mu, ak.firsts(tight_mu.phi), ak.full_like(met_pt, 0.0))), 0.0)
    sum_lep_px = lep_pt * np.cos(lep_phi)
    sum_lep_py = lep_pt * np.sin(lep_phi)
    recoil = recoil_pt(met_pt, met_phi, sum_lep_px, sum_lep_py)
    presel = (recoil > PRESEL_RECOIL_MIN) & (njets >= 1) & (lead_jet_pt > LEAD_JET_PT_MIN)
    dphi = np.abs(ak.to_numpy(met_phi) - ak.to_numpy(lep_phi))
    dphi = np.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    mt = np.sqrt(2.0 * ak.to_numpy(met_pt) * ak.to_numpy(lep_pt) * (1.0 - np.cos(dphi)))
    mt = ak.Array(mt)
    n_non_b = njets - nbjets
    return presel & (n_tight_leptons(events) == 1) & (lep_pt > LEAD_LEP_PT_CR) & (mt < TOP_CR_MT_MAX) & (nbjets >= 2) & (n_non_b >= 2)

def top_cr_recoil(events):
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele, tight_mu = select_tight_electrons(events), select_tight_muons(events)
    one_ele = (ak.count(tight_ele.pt, axis=1) == 1) & (ak.count(tight_mu.pt, axis=1) == 0)
    one_mu = (ak.count(tight_ele.pt, axis=1) == 0) & (ak.count(tight_mu.pt, axis=1) == 1)
    lep_pt = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.pt), ak.where(one_mu, ak.firsts(tight_mu.pt), ak.full_like(met_pt, 0.0))), 0.0)
    lep_phi = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.phi), ak.where(one_mu, ak.firsts(tight_mu.phi), ak.full_like(met_pt, 0.0))), 0.0)
    return recoil_pt(met_pt, met_phi, lep_pt * np.cos(lep_phi), lep_pt * np.sin(lep_phi))

fig, ax = plt.subplots()
for name in list(events_by_dataset.keys())[:5]:
    ev = events_by_dataset[name]
    mask = top_cr_mask(ev)
    recoil = top_cr_recoil(ev)[mask]
    if len(ak.ravel(recoil)) == 0:
        continue
    ax.hist(ak.to_numpy(ak.ravel(recoil)), bins=30, range=(250, 600), label=latex_labels.get(name, labels.get(name, name)), alpha=0.6, histtype="step")
ax.set_xlabel("Recoil [GeV]")
ax.set_ylabel("Events")
ax.set_title("Top CR (single-lepton): Recoil (data and MC)")
ax.legend()
plt.show()

## Comparison: MET presel vs SR (Ex 3.4)

In [ ]:
# Use first MC background for comparison plot
name = bkg_names[0] if bkg_names else list(events_by_dataset.keys())[0]
ev = events_by_dataset[name]
good_jets = select_good_jets(ev)
njets = ak.num(good_jets)
nbjets = count_bjets(good_jets)
nlep = n_tight_leptons(ev)
met = ev.MET.pt
sr_mask = (njets >= 1) & (nbjets >= 2) & (nlep == 0) & (met > MET_SR_MIN)
met_presel = met[njets >= 1]
met_sr = met[sr_mask]

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(ak.to_numpy(ak.ravel(met_presel)), bins=50, range=(0, 500), edgecolor="black", alpha=0.7, label="Presel")
ax[0].set_xlabel("MET [GeV]")
ax[0].set_ylabel("Events")
ax[0].set_title("MET (≥1 jet)")
ax[1].hist(ak.to_numpy(ak.ravel(met_sr)), bins=50, range=(200, 600), edgecolor="black", alpha=0.7, label="SR")
ax[1].set_xlabel("MET [GeV]")
ax[1].set_ylabel("Events")
ax[1].set_title("MET (signal region)")
plt.tight_layout()
plt.show()